In [1]:
import os
import numpy as np
import pandas as pd
from glob import glob
import random
from sklearn.model_selection import train_test_split


In [2]:
patches_dir = "../preprocessed/balanced_patches"  # Use the balanced patches
dataset_dir = "../preprocessed/dataset_splits"    # Where we'll save train/val/test splits
os.makedirs(dataset_dir, exist_ok=True)


3. Load and Label Patches

In [3]:
patch_files = glob(os.path.join(patches_dir, "*.npz"))

data = []

for path in patch_files:
    npz = np.load(path)
    label = int(npz['label'])
    data.append((path, label))

# Convert to DataFrame for easier handling
df = pd.DataFrame(data, columns=["path", "label"])

print(f"Total patches: {len(df)}")
print(df["label"].value_counts())


Total patches: 196
label
0    150
1     46
Name: count, dtype: int64


4. Stratified Train/Val/Test Split

In [4]:
# First split train + temp (val + test)
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=42
)

# Then split temp into val + test
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")


Train: 137
Validation: 29
Test: 30


5. Preprocess and Save Splits

In [5]:
def preprocess_and_save(split_df, split_name):
    split_folder = os.path.join(dataset_dir, split_name)
    os.makedirs(split_folder, exist_ok=True)

    for idx, row in split_df.iterrows():
        data = np.load(row['path'])

        patch = data['patch'].astype(np.float32)
        label = int(data['label'])

        # Normalize: Clip HU and scale to [0, 1]
        patch = np.clip(patch, -1000, 400)
        patch = (patch + 1000) / 1400  # because [-1000, 400] range -> [0, 1]

        # Save normalized patch
        filename = os.path.basename(row['path'])
        np.savez_compressed(os.path.join(split_folder, filename), patch=patch, label=label)

    print(f"✅ Saved {len(split_df)} patches to {split_name}/")

# Apply to each split
preprocess_and_save(train_df, "train")
preprocess_and_save(val_df, "val")
preprocess_and_save(test_df, "test")


✅ Saved 137 patches to train/
✅ Saved 29 patches to val/
✅ Saved 30 patches to test/
